# 🌍 Green Bond Thesis — ClimateBERT Sentiment Pipeline
**Title:** Do Politicians' Words Have a Dollar Value?  
**Author:** Nahian Ibnat  
**Supervisor:** Zoltán Csaba Tóth  
**Institution:** Central European University — MA in Economics, Data & Policy  
**Date:** April 2026

---

## What This Notebook Does

This notebook implements a two-stage ClimateBERT sentiment pipeline on real English-language article headlines fetched from the GDELT DOC API. It fixes the previous pipeline failure where ClimateBERT returned all-zero scores because it was applied to GDELT theme codes (structured keyword tags) rather than natural language text.

### Two-Stage Pipeline

| Stage | Model | Purpose |
|-------|-------|---------|
| **Stage 1** | `distilroberta-base-climate-detector` | Filter paragraphs to climate-relevant only |
| **Stage 2** | `distilroberta-base-climate-sentiment` | Classify sentiment: opportunity / neutral / risk |

> **Important:** ClimateBERT is trained on **paragraphs**, not single sentences. Headlines from the same country-day are concatenated into a paragraph before scoring.

### Setup Instructions
> 1. Go to **Runtime → Change runtime type → T4 GPU**  
> 2. Run all cells sequentially  
> 3. Ensure `daily_sentiment_gdelt.csv` exists in `Thesis/data/processed/gdelt/` on Google Drive  
> 4. Output files are saved directly to Google Drive


---
# Step 1 — Install, Import & Load ClimateBERT Models


## 1.1 — Install Dependencies

In [ ]:
!pip install -q transformers torch requests tqdm pandas numpy matplotlib

## 1.2 — Imports & Configuration

In [ ]:
import os
import time
import requests
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Mount Google Drive ─────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Thesis root on Drive ──────────────────────────────
ROOT = "/content/drive/MyDrive/Thesis"

# ── Output directories ────────────────────────────────
DATA_DIR   = os.path.join(ROOT, "data", "processed", "gdelt")   # CSVs
OUTPUT_DIR = os.path.join(ROOT, "output", "gdelt")              # PNGs
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Study parameters ───────────────────────────────────────────────────────────
COUNTRIES  = ['India', 'Indonesia']
START_YEAR = 2020
END_YEAR   = 2024

# ── Key political event dates ──────────────────────────────────────────────────
EVENTS = {
    '2022-11-15': 'JETP\n(G20)',
    '2023-01-25': 'India\nGreen Bond',
}

# ── Chart styling ──────────────────────────────────────────────────────────────
BG       = '#0f1117'
PANEL_BG = '#1a1d27'
TEAL     = '#1D9E75'
ORANGE   = '#E85D24'
GRAY     = '#8892a4'
WHITE    = '#e8eaf0'

def style_ax(ax):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=GRAY, labelsize=8)
    ax.xaxis.label.set_color(GRAY)
    ax.yaxis.label.set_color(GRAY)
    ax.title.set_color(WHITE)
    for s in ax.spines.values():
        s.set_edgecolor('#2a2d3a')

print(f"Data directory   : {DATA_DIR}")
print(f"Output directory : {OUTPUT_DIR}")
print(f"Countries        : {COUNTRIES}")
print(f"Period           : {START_YEAR}–{END_YEAR}")
print(f"GPU available    : {torch.cuda.is_available()}")


## 1.3 — Load ClimateBERT Models

Two models are loaded:
- **Detector** — classifies text as climate-relevant (`yes`) or not (`no`)
- **Sentiment** — classifies climate-relevant text as `opportunity`, `neutral`, or `risk`

> ⏱ First load takes 1–2 minutes while models download from HuggingFace.


In [ ]:
device = 0 if torch.cuda.is_available() else -1

# ── Stage 1: Climate Detector ──────────────────────────────────────────────────
DETECTOR_MODEL = "climatebert/distilroberta-base-climate-detector"
print(f"Loading detector: {DETECTOR_MODEL}...")
detector = pipeline(
    "text-classification",
    model=DETECTOR_MODEL,
    tokenizer=AutoTokenizer.from_pretrained(DETECTOR_MODEL),
    device=device,
    truncation=True,
    max_length=512,
)

# ── Stage 2: Sentiment Classifier ─────────────────────────────────────────────
SENTIMENT_MODEL = "climatebert/distilroberta-base-climate-sentiment"
print(f"Loading sentiment: {SENTIMENT_MODEL}...")
sentiment_clf = pipeline(
    "text-classification",
    model=SENTIMENT_MODEL,
    tokenizer=AutoTokenizer.from_pretrained(SENTIMENT_MODEL),
    device=device,
    truncation=True,
    max_length=512,
)

print(f"\n✓ Both models loaded | Device: {'GPU' if device == 0 else 'CPU'}")

# ── Sanity check on test paragraphs ───────────────────────────────────────────
test_paragraphs = [
    "India has announced an ambitious renewable energy programme targeting 500 GW of clean energy capacity by 2030. The government has committed substantial fiscal resources to accelerate the transition away from coal, signalling strong political will to meet its Paris Agreement commitments.",
    "Indonesia approved the expansion of several captive coal mining projects this quarter despite international pressure from the JETP partnership. Energy ministry officials indicated that coal will remain central to the domestic energy mix through 2040, contradicting climate commitments made at COP26.",
    "The central bank released its quarterly monetary policy statement today, keeping interest rates unchanged amid moderate inflationary pressures.",
]

print("\nSanity Check — Stage 1: Climate Detection")
print("-" * 55)
for text, r in zip(test_paragraphs, detector(test_paragraphs)):
    print(f"  [{r['label']:<5} {r['score']:.3f}] {text[:65]}...")

print("\nSanity Check — Stage 2: Sentiment Classification")
print("-" * 55)
for text, r in zip(test_paragraphs, sentiment_clf(test_paragraphs)):
    print(f"  [{r['label']:<12} {r['score']:.3f}] {text[:65]}...")


---
# Step 2 — Fetch Article Headlines from GDELT DOC API

Fetches real English-language article headlines about climate and energy policy in India and Indonesia.

**Strategy:**
- Half-year windows (2020-H1 through 2024-H2) for better date distribution
- 250 articles per window — maximum allowed by the API
- Progressive wait times and exponential backoff to handle rate limiting (HTTP 429)
- 90-second pause between countries to reset the rate limit window

> ⏱ **Estimated time: 15–20 minutes** — do not interrupt this cell.


In [ ]:
QUERIES = {
    'India':     'climate India government',
    'Indonesia': 'climate Indonesia government',
}

WINDOWS = []
for year in range(2020, 2025):
    WINDOWS.extend([
        (f"{year}0101000000", f"{year}0630235959", f"{year}-H1"),
        (f"{year}0701000000", f"{year}1231235959", f"{year}-H2"),
    ])

def fetch_with_backoff(params, max_attempts=6):
    """Fetch from GDELT DOC API with exponential backoff on rate limits."""
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    for attempt in range(max_attempts):
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code == 200:
                return r.json().get('articles', [])
            elif r.status_code == 429:
                wait = min(30 * (2 ** attempt) + random.uniform(0, 10), 300)
                print(f"    429 — waiting {wait:.0f}s (attempt {attempt+1}/{max_attempts})")
                time.sleep(wait)
            else:
                print(f"    HTTP {r.status_code} — waiting 15s...")
                time.sleep(15)
        except requests.exceptions.Timeout:
            print(f"    Timeout — waiting 20s...")
            time.sleep(20)
        except Exception as e:
            print(f"    Error: {e} — waiting 15s...")
            time.sleep(15)
    print(f"    ✗ Failed after {max_attempts} attempts — skipping")
    return []

all_articles = []
print(f"Fetching headlines: {len(WINDOWS)} windows × 2 countries = {len(WINDOWS)*2} requests")
print("Estimated time: 15–20 minutes\n")

for ci, country in enumerate(COUNTRIES):
    query = QUERIES[country]

    if ci > 0:
        print(f"\nPausing 90 seconds before {country} to reset rate limit...\n")
        time.sleep(90)

    print(f"{'='*50}")
    print(f"Country: {country}")
    print(f"{'='*50}")
    country_total = 0

    for wi, (start_dt, end_dt, label) in enumerate(WINDOWS):
        wait_time = 20 + (wi * 2) + random.uniform(0, 5)
        print(f"  [{label}] Waiting {wait_time:.0f}s...", end=' ', flush=True)
        time.sleep(wait_time)

        articles = fetch_with_backoff({
            "query":         query,
            "mode":          "artlist",
            "maxrecords":    250,
            "startdatetime": start_dt,
            "enddatetime":   end_dt,
            "format":        "json",
            "sort":          "DateDesc",
        })

        count = len(articles)
        country_total += count

        for art in articles:
            title = art.get('title', '').strip()
            seen  = art.get('seendate', '')
            if title and len(title) > 15:
                all_articles.append({
                    'country':  country,
                    'date_str': seen[:8] if len(seen) >= 8 else '',
                    'title':    title,
                    'url':      art.get('url', ''),
                    'domain':   art.get('domain', ''),
                    'language': art.get('language', ''),
                })
        print(f"→ {count} articles (total: {country_total})")

    print(f"\n  ✓ {country} total: {country_total} articles\n")

# ── Build DataFrame ────────────────────────────────────────────────────────────
df = pd.DataFrame(all_articles)
df['date'] = pd.to_datetime(df['date_str'], format='%Y%m%d', errors='coerce')
df = df.dropna(subset=['date', 'title']).reset_index(drop=True)

# English only — ClimateBERT is trained on English
df_en = df[
    df['language'].str.lower().isin(['english', 'eng', '']) | df['language'].isna()
].drop_duplicates(subset=['title']).copy().reset_index(drop=True)

print(f"Total headlines fetched  : {len(df):,}")
print(f"English (deduplicated)   : {len(df_en):,}")
print()
print(df_en.groupby('country').agg(
    articles=('title', 'count'),
    start=('date', 'min'),
    end=('date', 'max'),
).to_string())

df_en.to_csv(f"{OUTPUT_DIR}/gdelt_headlines_raw.csv", index=False)
print(f"\n✓ Saved → {OUTPUT_DIR}/gdelt_headlines_raw.csv")


---
# Step 3 — Two-Stage ClimateBERT Scoring

**Why paragraphs, not sentences?**  
ClimateBERT is trained on corporate climate disclosure *paragraphs*. Single sentences produce unreliable scores. We therefore concatenate all headlines from the same country-day into one paragraph before scoring.

**Stage 1 — Climate Detection**  
Filters each daily paragraph to determine whether it is climate-relevant. Non-climate days are excluded from sentiment scoring and assigned a score of 0.0.

**Stage 2 — Sentiment Classification**  
Scores climate-relevant paragraphs as:
- `opportunity` → +1.0 (positive climate signal)
- `neutral`     →  0.0
- `risk`        → −1.0 (negative climate signal)

> ⏱ **Estimated time: 5–10 minutes on T4 GPU**


In [ ]:
BATCH_SIZE = 16

# ── Auto-detect label mapping ──────────────────────────────────────────────────
test_result = sentiment_clf(["India government announces new solar energy targets for 2030."])
test_label  = test_result[0]['label'].lower()
print(f"Model label detected: '{test_label}'")

if test_label in ['opportunity', 'risk', 'neutral']:
    LABEL_MAP = {"opportunity": 1.0, "neutral": 0.0, "risk": -1.0}
    print("Label mapping: opportunity=+1.0 | neutral=0.0 | risk=−1.0")
elif test_label in ['positive', 'negative', 'neutral']:
    LABEL_MAP = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}
    print("Label mapping: positive=+1.0 | neutral=0.0 | negative=−1.0")
else:
    LABEL_MAP = {}
    print(f"Unknown label '{test_label}' — all scores will be 0.0")

# ── Group headlines into daily paragraphs ─────────────────────────────────────
daily_texts = (
    df_en.groupby(['country', 'date'])['title']
    .apply(lambda x: '. '.join(x.tolist()))
    .reset_index()
    .rename(columns={'title': 'paragraph'})
)
daily_texts['article_count'] = (
    df_en.groupby(['country', 'date'])['title']
    .count()
    .reset_index(drop=True)
)

print(f"\nDaily paragraphs to score: {len(daily_texts):,}")
print(f"Sample (first 200 chars):")
print(f"  {daily_texts['paragraph'].iloc[0][:200]}...")

# ── Stage 1: Climate Detection ─────────────────────────────────────────────────
print(f"\nStage 1: Detecting climate-relevant paragraphs...")
paragraphs = daily_texts['paragraph'].tolist()
is_climate = []

for i in tqdm(range(0, len(paragraphs), BATCH_SIZE), desc="Climate detection"):
    batch = [str(t)[:512] for t in paragraphs[i:i+BATCH_SIZE]]
    try:
        results = detector(batch)
        is_climate.extend([r['label'].lower() == 'yes' for r in results])
    except Exception as e:
        print(f"  Batch error: {e}")
        is_climate.extend([True] * len(batch))

daily_texts['is_climate'] = is_climate
climate_count = sum(is_climate)
print(f"  Climate-relevant: {climate_count:,} / {len(daily_texts):,} "
      f"({climate_count/len(daily_texts)*100:.1f}%)")

# ── Stage 2: Sentiment Classification ─────────────────────────────────────────
print(f"\nStage 2: Scoring {climate_count:,} climate-relevant paragraphs...")

cb_scores      = [0.0]       * len(daily_texts)
cb_labels      = ['neutral'] * len(daily_texts)
cb_confidences = [0.0]       * len(daily_texts)

climate_indices = [i for i, v in enumerate(is_climate) if v]
climate_texts   = [paragraphs[i] for i in climate_indices]

error_count = 0
for batch_start in tqdm(range(0, len(climate_texts), BATCH_SIZE), desc="Sentiment scoring"):
    batch      = [str(t)[:512] for t in climate_texts[batch_start:batch_start+BATCH_SIZE]]
    batch_idxs = climate_indices[batch_start:batch_start+BATCH_SIZE]
    try:
        results = sentiment_clf(batch)
        for idx, r in zip(batch_idxs, results):
            label              = r['label'].lower()
            cb_labels[idx]     = label
            cb_scores[idx]     = LABEL_MAP.get(label, 0.0)
            cb_confidences[idx]= round(r['score'], 4)
    except Exception as e:
        error_count += 1

daily_texts['climatebert_score']      = cb_scores
daily_texts['climatebert_label']      = cb_labels
daily_texts['climatebert_confidence'] = cb_confidences

if error_count > 0:
    print(f"  ⚠ {error_count} batches failed — kept as neutral")

# ── Results ────────────────────────────────────────────────────────────────────
climate_days = daily_texts[daily_texts['is_climate']].copy()
print("\nClimateBERT Label Distribution (climate-relevant days only):")
print(climate_days['climatebert_label'].value_counts().to_string())
print(f"\nMean score by country:")
print(climate_days.groupby('country')['climatebert_score'].mean().round(4).to_string())
print(f"\nScore distribution:")
print(climate_days['climatebert_score'].describe().round(4).to_string())
print("\nSample scored paragraphs:")
for _, row in climate_days.sample(min(3, len(climate_days)), random_state=42).iterrows():
    print(f"  [{row['climatebert_label']:<12} {row['climatebert_score']:+.1f}]"
          f" {row['paragraph'][:90]}...")

daily_texts.to_csv(f"{DATA_DIR}/gdelt_climatebert_scored.csv", index=False)
print(f"\n✓ Scored data saved → {DATA_DIR}/gdelt_climatebert_scored.csv")


---
# Step 4 — Aggregate to Daily Country-Level Series

Produces the daily ClimateBERT sentiment variable:

$$\text{ClimateBERT}_{c,t} \in [-1, +1]$$

Only climate-relevant days are included. Non-climate days are excluded from the merge rather than set to zero, to avoid diluting the signal.


In [ ]:
daily_cb = daily_texts.copy()

# Rename for consistency with V2Tone naming convention
daily_cb = daily_cb.rename(columns={
    'climatebert_score':      'sentiment_climatebert',
    'climatebert_confidence': 'climatebert_confidence',
})

# Add percentage positive/negative
daily_cb['climatebert_pct_pos'] = (daily_cb['sentiment_climatebert'] > 0).astype(float)
daily_cb['climatebert_pct_neg'] = (daily_cb['sentiment_climatebert'] < 0).astype(float)

# Keep only climate-relevant days
daily_cb_clean = daily_cb[daily_cb['is_climate']].copy().reset_index(drop=True)

print("Daily ClimateBERT Series Summary")
print("=" * 60)
print(daily_cb_clean.groupby('country').agg(
    days           = ('date',                  'nunique'),
    mean_sentiment = ('sentiment_climatebert',  'mean'),
    mean_pct_pos   = ('climatebert_pct_pos',    'mean'),
    mean_pct_neg   = ('climatebert_pct_neg',    'mean'),
    total_articles = ('article_count',          'sum'),
).round(4).to_string())

daily_cb_clean.to_csv(f"{DATA_DIR}/daily_climatebert.csv", index=False)
print(f"\n✓ Saved → {DATA_DIR}/daily_climatebert.csv")


---
# Step 5 — Merge with Existing V2Tone Series

Upload your existing `daily_sentiment_gdelt.csv` file when prompted. This file is in your `Thesis/GDELT_data/` folder on Google Drive.

The ClimateBERT scores are merged as a sparse overlay — most days will have V2Tone scores but only ~50 days will have ClimateBERT scores. This is expected given the API coverage constraints.

**Key output:** The correlation between V2Tone and ClimateBERT validates V2Tone as a directional sentiment proxy.


In [ ]:
# ── Load V2Tone series directly from Drive ─────────────────────────────────
v2tone_path = os.path.join(DATA_DIR, "daily_sentiment_gdelt.csv")
print(f"Loading V2Tone from: {v2tone_path}")
assert os.path.exists(v2tone_path), f"File not found: {v2tone_path}"
print("✓ File found.")


In [ ]:
# ── Load V2Tone series ─────────────────────────────────────────────────────────
daily_v2    = pd.read_csv(v2tone_path, parse_dates=['date'])

# Drop any leftover climatebert columns from previous runs
drop_cols = [c for c in daily_v2.columns if 'climatebert' in c.lower()]
if drop_cols:
    print(f"Dropping stale columns from V2Tone file: {drop_cols}")
    daily_v2 = daily_v2.drop(columns=drop_cols)

print(f"V2Tone loaded: {len(daily_v2):,} rows | "
      f"{daily_v2.date.min().date()} → {daily_v2.date.max().date()}")

# ── Merge ──────────────────────────────────────────────────────────────────────
merge_cols = [c for c in ['country','date','sentiment_climatebert',
              'climatebert_pct_pos','climatebert_pct_neg',
              'climatebert_confidence','article_count','climatebert_label']
              if c in daily_cb_clean.columns]

daily_merged = pd.merge(
    daily_v2,
    daily_cb_clean[merge_cols],
    on=['country','date'],
    how='left'
)

# ── Coverage ───────────────────────────────────────────────────────────────────
print(f"\nMerged rows: {len(daily_merged):,}")
print(f"\nClimateBERT coverage:")
for country in COUNTRIES:
    sub     = daily_merged[daily_merged['country']==country]
    covered = sub['sentiment_climatebert'].notna().sum()
    print(f"  {country:<12}: {covered}/{len(sub)} days ({covered/len(sub)*100:.1f}%)")

# ── Correlation ────────────────────────────────────────────────────────────────
matched = daily_merged.dropna(subset=['sentiment_gdelt','sentiment_climatebert'])
print(f"\nCorrelation between V2Tone and ClimateBERT (n={len(matched)} matched days):")
print("-" * 50)
for country in COUNTRIES:
    sub = matched[matched['country']==country]
    if len(sub) > 1:
        corr = sub[['sentiment_gdelt','sentiment_climatebert']].corr().iloc[0,1]
        print(f"  {country:<12}: r = {corr:+.4f}")
if len(matched) > 1:
    corr_all = matched[['sentiment_gdelt','sentiment_climatebert']].corr().iloc[0,1]
    print(f"  {'Overall':<12}: r = {corr_all:+.4f}")

daily_merged.to_csv(f"{DATA_DIR}/daily_sentiment_merged.csv", index=False)
print(f"\n✓ Saved → {DATA_DIR}/daily_sentiment_merged.csv")


---
# Step 6 — Visualisation

Three charts saved to the output folder:

| Chart | Description |
|-------|-------------|
| `climatebert_timeline.png` | V2Tone (line) + ClimateBERT (scatter) by country |
| `climatebert_validation.png` | V2Tone vs ClimateBERT scatter with correlation |
| `climatebert_distribution.png` | ClimateBERT label distribution by country |


## 6.1 — Sentiment Timeline

In [ ]:
colors = {'India': ORANGE, 'Indonesia': TEAL}

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
fig.patch.set_facecolor(BG)
fig.suptitle('Daily Climate Sentiment — India & Indonesia (2020–2024)\n'
             'V2Tone (line) vs ClimateBERT (scatter)',
             fontsize=13, fontweight='bold', color=WHITE, y=1.01)

for ax, (country, color) in zip(axes, colors.items()):
    style_ax(ax)
    data = daily_merged[daily_merged['country']==country].copy().sort_values('date')

    # V2Tone — dense daily series as smooth line
    data['smooth_v2'] = data['sentiment_gdelt'].rolling(30, min_periods=5).mean()
    ax.plot(data['date'], data['smooth_v2'], color=GRAY, linewidth=1.2,
            linestyle='--', label='V2Tone (30d MA)', alpha=0.8, zorder=2)

    # ClimateBERT — sparse scores as scatter points
    cb = data.dropna(subset=['sentiment_climatebert'])
    ax.scatter(cb['date'], cb['sentiment_climatebert'], color=color,
               s=45, alpha=0.85, zorder=4, label=f'ClimateBERT (n={len(cb)})')

    for edate, elabel in EVENTS.items():
        ax.axvline(pd.Timestamp(edate), color=GRAY, linewidth=0.8,
                   linestyle=':', alpha=0.7)
        ax.text(pd.Timestamp(edate), 0.9, elabel, fontsize=7,
                color=GRAY, ha='center', va='bottom')

    ax.axhline(0, color=GRAY, linewidth=0.6, linestyle='--', alpha=0.5)
    ax.set_ylabel(country, fontsize=10, fontweight='bold', color=color)
    ax.set_ylim(-1.3, 1.3)
    ax.legend(fontsize=8, facecolor=PANEL_BG, edgecolor='#2a2d3a',
              labelcolor=WHITE, loc='upper right')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.xlabel('Date', color=GRAY)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/climatebert_timeline.png", dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()
print("✓ Timeline chart saved")


## 6.2 — V2Tone vs ClimateBERT Validation Scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('V2Tone vs ClimateBERT — Validation Scatter',
             fontsize=12, fontweight='bold', color=WHITE)

matched_df = daily_merged.dropna(subset=['sentiment_gdelt','sentiment_climatebert'])

for ax, (country, color) in zip(axes, colors.items()):
    style_ax(ax)
    sub  = matched_df[matched_df['country']==country]
    corr = sub[['sentiment_gdelt','sentiment_climatebert']].corr().iloc[0,1] if len(sub)>1 else float('nan')

    ax.scatter(sub['sentiment_gdelt'], sub['sentiment_climatebert'],
               alpha=0.5, s=30, color=color)
    if len(sub) > 1:
        z  = np.polyfit(sub['sentiment_gdelt'], sub['sentiment_climatebert'], 1)
        xr = np.linspace(sub['sentiment_gdelt'].min(), sub['sentiment_gdelt'].max(), 100)
        ax.plot(xr, np.poly1d(z)(xr), color=WHITE, linewidth=1.5)

    ax.axhline(0, color=GRAY, linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axvline(0, color=GRAY, linewidth=0.5, linestyle='--', alpha=0.5)
    ax.set_xlabel('GDELT V2Tone Sentiment', fontsize=8)
    ax.set_ylabel('ClimateBERT Score', fontsize=8)
    ax.set_title(f'{country}  |  r = {corr:+.3f}  (n={len(sub)})',
                 fontsize=10, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/climatebert_validation.png", dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()
print("✓ Validation scatter saved")


## 6.3 — ClimateBERT Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('ClimateBERT Label Distribution by Country',
             fontsize=12, fontweight='bold', color=WHITE)

for ax, (country, color) in zip(axes, colors.items()):
    style_ax(ax)
    sub          = daily_cb_clean[daily_cb_clean['country']==country]
    label_counts = sub['climatebert_label'].value_counts()
    bar_colors   = [TEAL, GRAY, ORANGE][:len(label_counts)]

    bars = ax.bar(label_counts.index, label_counts.values,
                  color=bar_colors, alpha=0.85, edgecolor='#2a2d3a')

    for bar, val in zip(bars, label_counts.values):
        pct = val / len(sub) * 100
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.3,
                f'{pct:.1f}%', ha='center', va='bottom',
                fontsize=9, color=WHITE)

    ax.set_title(f'{country}  (n={len(sub):,} climate-relevant days)',
                 fontsize=10, fontweight='bold', color=color)
    ax.set_ylabel('Days', fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/climatebert_distribution.png", dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()
print("✓ Distribution chart saved")


---
# Step 7 — Download Output Files

Download all output files. The key file to upload to Databricks is `daily_sentiment_merged.csv`.

| File | Purpose |
|------|---------|
| `daily_sentiment_merged.csv` | Main merged file — contains V2Tone + ClimateBERT |
| `daily_climatebert.csv` | ClimateBERT daily series only |
| `gdelt_climatebert_scored.csv` | Article-level scored data |
| `climatebert_timeline.png` | Section 5.2 chart |
| `climatebert_validation.png` | Section 5.2 validation |
| `climatebert_distribution.png` | Section 5.2 distribution |


In [ ]:
from google.colab import files

csv_files = [
    f"{DATA_DIR}/daily_sentiment_merged.csv",
    f"{DATA_DIR}/daily_climatebert.csv",
    f"{DATA_DIR}/gdelt_climatebert_scored.csv",
]
png_files = [
    f"{OUTPUT_DIR}/climatebert_timeline.png",
    f"{OUTPUT_DIR}/climatebert_validation.png",
    f"{OUTPUT_DIR}/climatebert_distribution.png",
]

print("Downloading output files...")
for fpath in csv_files + png_files:
    if os.path.exists(fpath):
        files.download(fpath)
        print(f"  ✓ {os.path.basename(fpath)}")
    else:
        print(f"  ✗ Not found: {os.path.basename(fpath)}")

print("\n✓ Done.")
print("\nFiles are also saved on Google Drive:")
print(f"  CSVs → {DATA_DIR}")
print(f"  PNGs → {OUTPUT_DIR}")
